In [3]:
import kagglehub
import pandas as pd
import os

# 1. 下载数据集
path = kagglehub.dataset_download("thedevastator/5700-luckin-coffee-stores-across-china")
print("数据集已下载到:", path)

# 2. 找到 CSV 文件
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
csv_file_path = os.path.join(path, csv_files[0])
    
# 读取原始数据
raw_df = pd.read_csv(csv_file_path)

C:\Users\93189\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据集已下载到: C:\Users\93189\.cache\kagglehub\datasets\thedevastator\5700-luckin-coffee-stores-across-china\versions\2


In [16]:
raw_df.head(10)

,index,loc_name,storeid,address_raw,city,work_time,RunDate,loc_type,brand,shop_info
0,0,泰安东岳大街店,(No.A0832),泰山区财源街道东岳大街258号,泰安,10:00-21:00,08/01/21 07:10,代运营,luckin coffee,NaN
1,1,泰安宝龙城市广场店,(No.A1595),泰山区泰前街道宝龙城市广场B区2幢1036号,泰安,10:00-21:00,08/01/21 07:10,代运营,luckin coffee,NaN
2,2,泰安万达广场店,(No.A1363),泰山区财源街道万达金街E区124号,泰安,08:00-22:00,08/01/21 07:10,代运营,luckin coffee,NaN
3,3,泰山科技学院店,(No.A1187),岱岳区山口镇学院西路8号山东科技大学泰山科技学院新校区东岳书院院内,泰安,09:00-20:00,08/01/21 07:10,代运营,luckin coffee,NaN
4,4,泰安吾悦金街店,(No.A1978),敬请期待!,泰安,08:00-18:00,08/01/21 07:10,代运营,luckin coffee,NaN
5,5,曲阜师大店,(No.A1834),敬请期待!,泰安,08:00-18:00,08/01/21 07:10,代运营,luckin coffee,NaN
6,6,南通如皋新城吾悦广场店,(No.A0861),如皋市如城街道惠政路5号新城吾悦广场金街4-103（肯德基旁向南18米）,如皋(南通),08:00-21:00,08/01/21 07:10,代运营,luckin coffee,NaN
7,7,如皋安定广场安定街店,(No.A0870),如皋市安定街5好安定广场艾安阁北侧绿叶童装旁,如皋(南通),07:00-23:00,08/01/21 07:10,代运营,luckin coffee,NaN
8,8,河源翔丰商业广场店,(No.A1315),源城区上城街道翔丰商业广场,河源,08:30-22:30,08/01/21 07:10,代运营,luckin coffee,NaN
9,9,河源万隆城店,(No.A1236),源城区中山大道218号万隆财富中心写字楼一层大堂L102,河源,08:30-22:30,08/01/21 07:10,代运营,luckin coffee,NaN


我们读取了Kaggle上的Luckin在中国的地址的数据，因为这份数据是人工录入的，且格式没有划分很细，所以非常适合拿来做SAP数据清晰的例子。

为了有动力做数据清洗，我们定义以下目标:

1.统一数据格式。目标生成数据包含：
    storeid （如SAP customer Number）
    country
    region（省份）
    city（地级市）
    district（县级市/区）
    address（具体地址）
2.数据清洗
3.使用地图生成经纬度

In [4]:
import requests
import csv

def generate_region_table():
    url = "https://cdn.jsdelivr.net/gh/modood/Administrative-divisions-of-China@master/dist/pca-code.json"
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"数据下载失败，请检查网络连接: {e}")
        return

    filename = 'china_regions_complete.csv'
    
    with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['省份名称', '省份代码', '地级市名称', '城市代码', '区县名称', '区县代码'])
        
        count = 0
        # 这次 data 是一个标准的列表(list)，内部包含字典
        for province in data:
            p_name = province.get('name', '')
            p_code = province.get('code', '')
            
            for city in province.get('children', []):
                c_name = city.get('name', '')
                c_code = city.get('code', '')
                
                for county in city.get('children', []):
                    co_name = county.get('name', '')
                    co_code = county.get('code', '')
                    
                    writer.writerow([p_name, p_code, c_name, c_code, co_name, co_code])
                    count += 1
                    
    print(f"表格生成完毕！共成功写入 {count} 条数据。")
    print(f"请在当前目录下查看文件：{filename}")

if __name__ == "__main__":
    generate_region_table()

表格生成完毕！共成功写入 3056 条数据。
请在当前目录下查看文件：china_regions_complete.csv


In [10]:
counts = raw_df['address_raw'].value_counts()
frequent_values = counts[counts > 2]
print(frequent_values)

address_raw
敬请期待!    248
Name: count, dtype: int64


In [9]:
Reduced_df = raw_df[['loc_name', 'storeid', 'address_raw','city']].head(20)
Reduced_df

,loc_name,storeid,address_raw,city
0,泰安东岳大街店,(No.A0832),泰山区财源街道东岳大街258号,泰安
1,泰安宝龙城市广场店,(No.A1595),泰山区泰前街道宝龙城市广场B区2幢1036号,泰安
2,泰安万达广场店,(No.A1363),泰山区财源街道万达金街E区124号,泰安
3,泰山科技学院店,(No.A1187),岱岳区山口镇学院西路8号山东科技大学泰山科技学院新校区东岳书院院内,泰安
4,泰安吾悦金街店,(No.A1978),敬请期待!,泰安
5,曲阜师大店,(No.A1834),敬请期待!,泰安
6,南通如皋新城吾悦广场店,(No.A0861),如皋市如城街道惠政路5号新城吾悦广场金街4-103（肯德基旁向南18米）,如皋(南通)
7,如皋安定广场安定街店,(No.A0870),如皋市安定街5好安定广场艾安阁北侧绿叶童装旁,如皋(南通)
8,河源翔丰商业广场店,(No.A1315),源城区上城街道翔丰商业广场,河源
9,河源万隆城店,(No.A1236),源城区中山大道218号万隆财富中心写字楼一层大堂L102,河源


In [13]:
import os
import time
import requests
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

# 激活 pandas 的进度条支持
tqdm.pandas(desc="调用高德API处理进度")

# ================= 1. 加载并获取环境变量 =================
load_dotenv() 

AMAP_API_KEY = os.getenv("AMAP_API_KEY")

if not AMAP_API_KEY:
    raise ValueError("未找到 AMAP_API_KEY，请检查 .env 文件是否配置正确，以及是否与脚本在同一目录下！")

# ================= 2. 定义代码 =================
def fetch_amap_all_data(row):
    """
    请求高德API并返回所有的字段数据
    """
    # 增加一个极小的延迟，防止请求过快被高德API限制或封IP
    time.sleep(0.05) 
    
    city = row['city']
    address_raw = row['address_raw']
    loc_name = row['loc_name']
    
    # 1. 确定搜索关键词
    if address_raw != '敬请期待！':
        keyword = address_raw
    else:
        keyword = f"瑞幸{loc_name}"
        
    # 2. 构造高德API请求
    url = "https://restapi.amap.com/v3/place/text"
    params = {
        'key': AMAP_API_KEY,
        'keywords': keyword,
        'city': city,
        'offset': 1,    # 只取最匹配的第一条结果
        'page': 1,
        'extensions': 'base'  
    }
    
    # 3. 发送请求并全量提取数据
    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        
        if data.get('status') == '1' and int(data.get('count', 0)) > 0:
            poi = data['pois'][0]
            
            cleaned_poi = {}
            for k, v in poi.items():
                if isinstance(v, list):
                    if len(v) == 0:
                        cleaned_poi[k] = ""
                    else:
                        cleaned_poi[k] = ",".join([str(i) for i in v])
                elif isinstance(v, dict):
                    cleaned_poi[k] = str(v)
                else:
                    cleaned_poi[k] = v
                    
            cleaned_poi['search_keyword_used'] = keyword 
            
            return pd.Series(cleaned_poi)
            
    except Exception as e:
        print(f"请求出错: keyword={keyword}, error={e}")
        
    # 如果没查到或者报错，返回空Series
    return pd.Series(dtype=object)

# ================= 3. 执行与清洗数据 =================

# 提示：确保此时你的 Reduced_df 已经定义并加载了数据
# 例如：Reduced_df = pd.read_csv("data.csv")

print(f"开始处理，共计 {len(Reduced_df)} 条数据...")

api_result_columns = Reduced_df.progress_apply(fetch_amap_all_data, axis=1)
df_final = pd.concat([Reduced_df, api_result_columns], axis=1)

开始处理，共计 20 条数据...


调用高德API处理进度: 100%|██████████| 20/20 [00:28<00:00,  1.44s/it]


处理完成！文件已保存至当前目录下的 amap_full_data_result.xlsx。


,loc_name,storeid,address_raw,city,parent,distance,keytag,importance,biz_ext,type,...,biz_type,cityname,childtype,atag,name,location,shopid,favorite_num,featured_reviews,search_keyword_used
0,泰安东岳大街店,(No.A0832),泰山区财源街道东岳大街258号,泰安,,,SPA,,,生活服务;洗浴推拿场所;洗浴推拿场所,...,,泰安市,,,金城足道按摩SPA(岱庙店),"117.123737,36.190328",,,,泰山区财源街道东岳大街258号
1,泰安宝龙城市广场店,(No.A1595),泰山区泰前街道宝龙城市广场B区2幢1036号,泰安,B021A0O09O,,咖啡,,,餐饮服务;咖啡厅;咖啡厅,...,diner,泰安市,202,"美式,拿铁,海盐芝士厚乳拿铁,杨枝甘露瑞纳冰,生椰拿铁yyds,厚乳拿铁,生椰拿铁,杨枝甘露",瑞幸咖啡(泰安宝龙城市广场店),"117.150254,36.207768",,,,泰山区泰前街道宝龙城市广场B区2幢1036号
2,泰安万达广场店,(No.A1363),泰山区财源街道万达金街E区124号,泰安,B0KUNZLABP,,"政府机构及社会团体,政府及社会团体相关,政府及社会团体相关",,,政府机构及社会团体;政府及社会团体相关;政府及社会团体相关,...,,泰安市,202,,泰安区财源街道万达商圈党群服务中心,"117.087329,36.178591",,,,泰山区财源街道万达金街E区124号
3,泰山科技学院店,(No.A1187),岱岳区山口镇学院西路8号山东科技大学泰山科技学院新校区东岳书院院内,泰安,B0G01ZU0FR,,咖啡,,,餐饮服务;咖啡厅;咖啡厅,...,diner,泰安市,320,,瑞幸咖啡(泰山科技学院店),"117.288412,36.242004",,,,岱岳区山口镇学院西路8号山东科技大学泰山科技学院新校区东岳书院院内
4,泰安吾悦金街店,(No.A1978),敬请期待!,泰安,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
#因为手动搜索几个为NAN的数据，发现瑞幸地址搜不到。所以估计是没开起来。故此可以考虑删除所有NAN的数据。
df_final = df_final.dropna(subset = ['address'])
df_final.info()

<class 'pandas.DataFrame'>
Index: 17 entries, 0 to 19
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   loc_name                 17 non-null     str   
 1   storeid                  17 non-null     str   
 2   address_raw              17 non-null     str   
 3   city                     17 non-null     str   
 4   parent                   17 non-null     object
 5   distance                 17 non-null     object
 6   keytag                   17 non-null     object
 7   importance               17 non-null     object
 8   biz_ext                  17 non-null     object
 9   type                     17 non-null     object
 10  photos                   17 non-null     object
 11  building                 17 non-null     object
 12  typecode                 17 non-null     object
 13  shopinfo                 17 non-null     object
 14  poiweight                17 non-null     object
 15  adname 

原本想用Chinese_City_List作为标准做一遍数据清洗，结果发现数据库里的数据已经满足需求并且很干净了。所以跳过，直接把数据整理的漂亮一点导出。

In [22]:
df0 = df_final[['storeid','pname','cityname','adname','address','location']]
df0 = df0.dropna(subset = ['address'])
df0[['longitude', 'latitude']] = df0['location'].str.split(',', expand=True)
df0 = df0.rename(columns={
    'storeid': 'BP code', 
    'pname': 'Province(SAP Field REGION)',
    'cityname': 'City(SAP Field CITY1)',
    'adname': 'District(SAP Field CITY2)',
    'address': 'Address(SAP Field STREET)'
})

df0.head(10)

,BP code,Province(SAP Field REGION),City(SAP Field CITY1),District(SAP Field CITY2),Address(SAP Field STREET),location,longitude,latitude
0,(No.A0832),山东省,泰安市,泰山区,东岳大街258号,"117.123737,36.190328",117.123737,36.190328
1,(No.A1595),山东省,泰安市,泰山区,泰前街道宝龙城市广场B区2幢1036号,"117.150254,36.207768",117.150254,36.207768
2,(No.A1363),山东省,泰安市,泰山区,万达金街A区124号,"117.087329,36.178591",117.087329,36.178591
3,(No.A1187),山东省,泰安市,岱岳区,山口镇学院西路8号山东科技大学泰山科技学院新校区东岳书院院内,"117.288412,36.242004",117.288412,36.242004
6,(No.A0861),江苏省,南通市,如皋市,如城街道惠政路5号新城吾悦广场4-103(吾悦广场1号门肯德基旁边金街往南18米),"120.577352,32.368666",120.577352,32.368666
7,(No.A0870),江苏省,南通市,如皋市,安定街5号,"120.565115,32.387439",120.565115,32.387439
8,(No.A1315),广东省,河源市,源城区,中山大道,"114.702390,23.738993",114.702390,23.738993
9,(No.A1236),广东省,河源市,源城区,中山大道218号万隆财富中心写字楼一层大堂L102,"114.706625,23.757423",114.706625,23.757423
10,(No.A1338),广东省,河源市,源城区,越王大道西侧永祥路北侧,"114.718024,23.768147",114.718024,23.768147
11,(No.A1299),广东省,河源市,源城区,源南镇胜利上门七娘寨地段东环路西面广晟学府花园汇华苑B7栋0011号正,"114.738101,23.750855",114.738101,23.750855


In [ ]:
output_filename = "Processed Data.xlsx"
df0.to_excel(output_filename, index=False)